# Homework Starter — Stage 04: Data Acquisition and Ingestion
Name:
Date:

## Objectives
- API ingestion with secrets in `.env`
- Scrape a permitted public table
- Validate and save raw data to `data/raw/`

In [20]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
!pip install pandas
!pip install requests
!pip install yfinance
!pip install python-dotenv
!pip install beautifulsoup4

In [21]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/yangzixuan/bootcamp_kathy_yang/homework/homework04

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [22]:
import os, pathlib, datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
load_dotenv(); print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

ALPHAVANTAGE_API_KEY loaded? True


## Helpers (use or modify)

In [23]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k,v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape, 'na_total': int(df.isna().sum().sum())}

## Part 1 — API Pull (Required)
Choose an endpoint (e.g., Alpha Vantage or use `yfinance` fallback).

In [24]:
SYMBOL = 'AAPL'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY'))
if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {'function':'TIME_SERIES_DAILY','symbol':SYMBOL,'outputsize':'compact','apikey':os.getenv('ALPHAVANTAGE_API_KEY')}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    # Alpha Vantage answers 200 OK with a prose blob when the free daily cap (25 calls)
    # is hit, so check for the series rather than trusting the status code.
    key = [k for k in js if 'Time Series' in k]
    if not key:
        print('Alpha Vantage returned no series:', str(list(js.values())[0])[:150])
        USE_ALPHA = False

if USE_ALPHA:
    df_api = pd.DataFrame(js[key[0]]).T.reset_index().rename(columns={'index':'date','4. close':'close'})[['date','close']]
    df_api['date'] = pd.to_datetime(df_api['date']); df_api['close'] = pd.to_numeric(df_api['close'])

if not USE_ALPHA:
    import yfinance as yf
    df_api = yf.download(SYMBOL, period='3mo', interval='1d', auto_adjust=False,
                         multi_level_index=False).reset_index()[['Date','Close']]
    df_api.columns = ['date','close']

v_api = validate(df_api, ['date','close']); v_api

{'missing': [], 'shape': (100, 2), 'na_total': 0}

In [25]:
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

Saved data/raw/api_source-alpha_symbol-AAPL_20260825-154633.csv


## Part 2 — Scrape a Public Table (Required)
Replace `SCRAPE_URL` with a permitted page containing a simple table.

In [26]:
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'  # TODO: replace with permitted page
headers = {'User-Agent':'AFE-Homework/1.0 (student practice, contact: zy2586@nyu.edu)'}
try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30); resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    table = soup.find('table', id='constituents')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th','td'])] for tr in soup.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)
except Exception as e:
    print('Scrape failed, using inline demo table:', e)
    html = '<table><tr><th>Ticker</th><th>Price</th></tr><tr><td>AAA</td><td>101.2</td></tr></table>'
    soup = BeautifulSoup(html, 'html.parser')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th','td'])] for tr in table.find_all('tr')]
    header, *data = [r for r in rows if r]
    data = [r for r in data if len(r) == len(header)]  # Filter out the "mixed-in" rows that do not match the column count (such as nested navigation boxes)
    df_scrape = pd.DataFrame(data, columns=header)

if 'Price' in df_scrape.columns:
    df_scrape['Price'] = pd.to_numeric(df_scrape['Price'], errors='coerce')
    
df_scrape = df_scrape.dropna(subset=['GICSSector']).reset_index(drop=True)
v_scrape = validate(df_scrape, list(df_scrape.columns)); v_scrape

{'missing': [], 'shape': (503, 8), 'na_total': 0}

In [27]:
df_scrape.head()

,Symbol,Security,GICSSector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,0000066740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,0000091142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,0000001800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,0001551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,0001467373,1989


In [28]:
df_scrape.isna().sum()

Symbol                   0
Security                 0
GICSSector               0
GICS Sub-Industry        0
Headquarters Location    0
Date added               0
CIK                      0
Founded                  0
dtype: int64

In [29]:
df_scrape[df_scrape['GICSSector'].isna()]

,Symbol,Security,GICSSector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded


In [30]:
_ = save_csv(df_scrape, prefix='scrape', site='wikipedia', table='sp500')

Saved data/raw/scrape_site-wikipedia_table-sp500_20260825-154633.csv


## Documentation

- **API Source**: Alpha Vantage `TIME_SERIES_DAILY` endpoint (`https://www.alphavantage.co/query`), 
  symbol=AAPL, outputsize=compact (~100 most recent trading days). API key loaded from `.env` 
  (not committed; see `.gitignore`). Fallback to `yfinance` (period=3mo) is implemented in case 
  the Alpha Vantage free-tier daily cap (25 calls/day) is hit.

- **Scrape Source**: Wikipedia, "List of S&P 500 companies" 
  (`https://en.wikipedia.org/wiki/List_of_S%26P_500_companies`), the table with `id='constituents'`.

- **Assumptions & risks**:
  - *Rate limits*: Alpha Vantage free tier allows 25 calls/day; it returns HTTP 200 with a prose 
    message (not an error) when the cap is hit, so the code checks for the presence of a 
    "Time Series" key rather than trusting the status code alone.
  - *Selector fragility*: The Wikipedia page contains a nested navigation table (a "vte" template) 
    embedded near the constituents table. A naive `find_all('tr')` on the outer table picked up 
    12 extra rows from this nested table, producing NaNs in 6 columns (confirmed via 
    `df_scrape.isna().sum()` and `df_scrape[df_scrape['GICSSector'].isna()]`). These rows were 
    removed with `dropna(subset=['GICSSector'])`, bringing the shape from (515, 8) to the correct 
    (503, 8) with 0 remaining NAs. This selector is fragile — if Wikipedia changes the page 
    structure again, the same issue (or a new one) could reappear, so re-validating shape/NA 
    counts after any re-run is recommended.
  - *Schema changes*: Both API and scrape sources are third-party and can change format/columns 
    without notice; code defends against this by validating required columns and dtypes rather 
    than assuming the shape from memory.

- **`.env` confirmed not committed**: verified via `git check-ignore -v .env`, which confirmed 
  the file is excluded by `homework/homework04/.gitignore`.